In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from pynq import Overlay
import time

# 加载 base overlay
ol = Overlay("base.bit")
ol.download()

# 设置 LED 为输出
ol.leds_gpio.write(0x04, 0x0)

# 打开摄像头
cap = cv2.VideoCapture(0)

# 读取第一帧作为背景
ret, bg_frame = cap.read()
if not ret:
    print("摄像头打开失败")
    cap.release()
else:
    bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
    print("开始运行，请在摄像头前移动...")
    print("点击 Jupyter 工具栏的 ⏹ 按钮停止")

    try:
        for i in range(100):
            ret, frame = cap.read()
            if not ret:
                break
            
            # 1. 灰度化
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            # 2. 帧差
            diff = cv2.absdiff(bg_gray, gray)
            
            # 3. 二值化
            _, binary = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
            
            # 4. 形态学去噪
            kernel = np.ones((5, 5), np.uint8)
            binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
            binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
            
            # 5. 找轮廓
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            # 6. 画框 + 计数
            target_count = 0
            for cnt in contours:
                if cv2.contourArea(cnt) > 500:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    target_count += 1
            
            # 7. 显示数量
            cv2.putText(frame, f"Targets: {target_count}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # 8. LED 报警逻辑
            if target_count > 0:
                ol.leds_gpio.write(0x00, 0x1)   # 有人 → LED0 亮
            else:
                ol.leds_gpio.write(0x00, 0x0)   # 没人 → LED 灭
            
            # 9. 更新背景
            bg_gray = gray.copy()
            
            # 10. 显示画面
            clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
            
    except KeyboardInterrupt:
        print("手动停止")

    # 清理
    ol.leds_gpio.write(0x00, 0x0)
    cap.release()
    print("程序结束")


手动停止
程序结束
